# Interruptible MLE agent: one run on Kaggle

Starts one run, or resumes it if an earlier version was cut off. Use **Save Version → Save & Run All**:
the notebook then runs in the background for up to about 12 hours with the browser closed, and
everything in `/kaggle/working` (the run folder) is kept as the version's output. See `docs/cloud-runs.md`.

**Before the first run**
- *Settings*: Accelerator **None** (CPU), Internet **On** (needs a phone-verified account).
- *Add-ons → Secrets*: `GITHUB_TOKEN` (fine-grained, read-only *Contents* on this repo), and
  `OPENROUTER_API_KEY` and/or `GEMINI_API_KEY`.
- A real task (`SMOKE = False`) needs role 3's evaluator and the prepared data, attached as a private
  Kaggle Dataset. Until then, keep `SMOKE = True`.

**To resume a run that was cut off**: attach the earlier version's output as an input and set
`RESTORE_FROM` to its `runs` folder, then save a new version. Docker is not available, so scripts run
with `--sandbox subprocess` (no isolation); the report states this.

In [ ]:
# Settings: the only cell to edit.
CONDITION = "configs/conditions/c1_autonomous.yaml"
TASK = "spooky-author-identification"
SEED = 0
PROVIDER = "openrouter"   # openrouter (experiments) | gemini (cheap testing)
BRANCH = "main"           # "nabil-coreagent" until PR #2 is merged
SMOKE = True              # True: toy task, no evaluator or Kaggle data needed. False: the real task
SMOKE_NODES = 20
DRY_RUN = False           # with SMOKE: canned replies, no API key needed
RESTORE_FROM = ""          # e.g. "/kaggle/input/<this-notebook>/runs" to continue a cut-off run

In [ ]:
import os
import subprocess
import sys

REPO = "github.com/nabilrakaiza/interruptible-mle-agent.git"


def git(*args, token=""):
    """Run git without ever printing the token, even when it fails."""
    result = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError("git failed: " + message)


def get_code(dest, token):
    """Clone the private repo (or update it), then remove the token from .git/config."""
    url = f"https://x-access-token:{token}@{REPO}"
    if not os.path.exists(dest):
        git("clone", "-q", "--branch", BRANCH, url, dest, token=token)
    else:
        git("-C", dest, "fetch", "-q", url, BRANCH, token=token)
        git("-C", dest, "checkout", "-q", "FETCH_HEAD", token=token)
    git("-C", dest, "remote", "set-url", "origin", f"https://{REPO}")


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["OPENROUTER_API_KEY", "GEMINI_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:  # noqa: BLE001 - the platforms raise their own types; not every key is needed
        print(f"{name} is not in Kaggle secrets; fine unless PROVIDER needs it")

CODE = "/tmp/imle"  # outside /kaggle/working, so the code is not saved as output
get_code(CODE, secrets.get_secret("GITHUB_TOKEN"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[sandbox,analysis]"], cwd=CODE, check=True)

In [ ]:
RUNS_ROOT = "/kaggle/working/runs"
SYNC = ["--restore-from", RESTORE_FROM] if RESTORE_FROM else []

cmd = [sys.executable, "scripts/cloud_run.py", "--seed", str(SEED), "--runs-root", RUNS_ROOT]
cmd += SYNC
if SMOKE:
    cmd += ["--smoke", "--nodes", str(SMOKE_NODES)] + (["--dry-run"] if DRY_RUN else ["--provider", PROVIDER])
else:
    cmd += ["--config", CONDITION, "--task", TASK, "--provider", PROVIDER]
print(" ".join(cmd))
# Stream the run's output into this cell as it happens.
process = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
print("exit code", process.wait())

## Look at the result

The trajectory figure and timeline are written into the run folder, so they are part of the output.

In [ ]:
import glob

for run in sorted(glob.glob(f"{RUNS_ROOT}/*")):
    subprocess.run([sys.executable, "scripts/trajectory.py", run], cwd=CODE, check=False, capture_output=True)
    print(run, "->", f"{run}/trajectory/")